In [1]:
from sls_client import get_sls_data_by_query
from datetime import datetime, timedelta
import pandas as pd


import sqlite3

def get_user_variant_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_variant.db"
    table_name = f"search_ab_user_variant_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    query = f"""
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{{digit}}') as api,pageName as page_ame,
json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].experimentId') experiment_id,
type,uid,date_format(__time__, '%Y%m%d') as ds,count(1) search_times,
array_join(array_agg(distinct json_extract_scalar(json_extract_scalar(ai, '$.qh.xm-ab-exp'),'$[1].variantId')),',') variant_list
from log group by 1,2,3,4,5,6 limit 1000000"""
    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    _df["variant_list"] = _df["variant_list"].fillna("none")

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    return _df


all_user_variant_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    df = get_user_variant_of_date_from_sls(current_date)
    all_user_variant_df = pd.concat([all_user_variant_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_variant_df.head(10)

即将获取数据: =====> 2025-01-01 00:00:00 2025-01-01 23:59:59.999999 xm-mall: 
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{digit}') as a
>=====数条数:7449
即将获取数据: =====> 2025-01-02 00:00:00 2025-01-02 23:59:59.999999 xm-mall: 
type:a and ap:/product/ and pageName:/search/goods|
select regexp_replace(ap, '\d+','{digit}') as a
>=====数条数:6864


,api,page_ame,experiment_id,type,uid,ds,search_times,variant_list
0,/product/{digit}/{digit},/search/goods,product_search_rerank,a,283899,20250101,19,V1
1,/product/{digit}/{digit},/search/goods,product_search_rerank,a,423422,20250101,8,V3
2,/product/{digit}/{digit},/search/goods,product_search_rerank,a,486298,20250101,4,V3
3,/product/{digit}/{digit},/search/goods,product_search_rerank,a,471910,20250101,14,V3
4,/product/{digit}/{digit},/search/goods,product_search_rerank,a,337420,20250101,8,V1
5,/product/{digit}/{digit},/search/goods,product_search_rerank,a,369843,20250101,1,V5
6,/product/{digit}/{digit},/search/goods,product_search_rerank,a,461869,20250101,8,V2
7,/product/{digit}/{digit},/search/goods,product_search_rerank,a,100144,20250101,29,V3
8,/product/{digit}/{digit},/search/goods,product_search_rerank,a,29319,20250101,3,V4
9,/product/{digit}/{digit},/search/goods,product_search_rerank,a,296591,20250101,6,V5


In [3]:
# idx:4,name:徐州奶油草莓 净重3-3.2斤/一级/单果10g+/板装,pid:goods,sku:5442468008,salePrice:69.5,pdid:702,stock:10000,ext:cross;idx:5,name:徐州奶油草莓 280G*1盒/一级/4*6/ ,pid:goods,sku:5442468073,salePrice:16.5,pdid:702,stock:10000,ext:cross;idx:6,name:徐州奶油草莓 净重2.8-3斤/一级/单果10g+/ 10盒,pid:goods,sku:5442468518,salePrice:74.5,pdid:702,stock:10000,ext:cross

view_query="""
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:(\d+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,
* from(
select uid,date_format(__time__, '%Y%m%d') ds,url,bid_list.sku_item,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)
having pid = 'goods'
"""

def get_user_sku_view_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_view.db"
    table_name = f"user_sku_view_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=view_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    return _df

all_user_sku_view_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    df = get_user_sku_view_of_date_from_sls(current_date)
    all_user_sku_view_df = pd.concat([all_user_sku_view_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_view_df.head(10)

即将获取数据: =====> 2025-01-01 00:00:00 2025-01-01 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  
>=====数条数:253492
即将获取数据: =====> 2025-01-02 00:00:00 2025-01-02 23:59:59.999999 xm-mall: 
type:view and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  
>=====数条数:243191


,idx,name,sku,pid,pdid,uid,ds,url,sku_item,search_query,rq_count,type
0,12,黑海盗纯牛奶 1L*12盒,607745051124,goods,11951,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:12,name:黑海盗纯牛奶 1L*12盒,pid:goods,sku:607745...",牛奶,771,view
1,13,新希望纯牛奶(白盒装) 1L*12盒,607187855103,goods,4353,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:13,name:新希望纯牛奶(白盒装) 1L*12盒,pid:goods,sku:6...",牛奶,772,view
2,14,伊利全脂纯牛奶,607336647442,goods,8552,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:14,name:伊利全脂纯牛奶,pid:goods,sku:607336647442...",牛奶,773,view
3,15,Protag纯牛奶 1L*1盒,607164503874,goods,4869,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:15,name:Protag纯牛奶 1L*1盒,pid:goods,sku:6071...",牛奶,774,view
4,16,澳兰克全脂纯牛奶 1L*1盒,607684760264,goods,8068,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:16,name:澳兰克全脂纯牛奶 1L*1盒,pid:goods,sku:60768...",牛奶,774,view
5,17,新希望纯牛奶(白盒装) 1L*1盒,607187855770,goods,4353,236417,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:17,name:新希望纯牛奶(白盒装) 1L*1盒,pid:goods,sku:60...",牛奶,775,view
6,0,爱乐薇(铁塔)淡奶油 1L*1盒,52106,goods,52,368419,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:0,name:爱乐薇(铁塔)淡奶油 1L*1盒,pid:goods,sku:5210...",铁塔,42,view
7,1,爱乐薇(铁塔)淡奶油 1L*12盒,null,goods,52,368419,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:1,name:爱乐薇(铁塔)淡奶油 1L*12盒,pid:goods,sku:N00...",铁塔,42,view
8,2,爱乐薇（粉塔）马斯卡波尼稀奶油 1L*1盒,15126316653,goods,2644,368419,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:2,name:爱乐薇（粉塔）马斯卡波尼稀奶油 1L*1盒,pid:goods,sku...",铁塔,42,view
9,3,爱乐薇(紫塔)超高温灭菌稀奶油 1L*1盒,60553221733,goods,2938,368419,20250101,https://h5.summerfarm.net/home.html?token=mall...,"idx:3,name:爱乐薇(紫塔)超高温灭菌稀奶油 1L*1盒,pid:goods,sku...",铁塔,42,view


In [5]:
click_query="""
type:cl and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  regexp_extract(sku_item, 'name:([^,]+)', 1) AS name,
  regexp_extract(sku_item, 'sku:(\d+)', 1) AS sku,
  regexp_extract(sku_item, 'pid:([^,]+)', 1) AS pid,
  regexp_extract(sku_item, 'pdid:(\d+)', 1) AS pdid,
ds,search_query,type,uid,page_name from(
select uid,date_format(__time__, '%Y%m%d') ds,url,bid_list.sku_item,pageName as page_name,
url_extract_parameter(split_part(url,'#/',2), 'pdName') AS search_query,rqCount rq_count,type
from log,unnest(split(bid,';')) as bid_list(sku_item) limit 10000000)"""

def get_user_sku_click_of_date_from_sls(
    day: datetime, check_if_local_exist: bool = True
) -> pd.DataFrame:
    db_file_name = f"./data/search_ab_user_sku_click.db"
    table_name = f"user_sku_click_{day.strftime('%Y%m%d')}"
    conn = sqlite3.connect(db_file_name)
    cursor = conn.cursor()

    if check_if_local_exist:
        try:
            query = f"SELECT * FROM {table_name}"
            df = pd.read_sql_query(query, conn)
            conn.close()
            return df
        except pd.io.sql.DatabaseError:
            pass

    from_time = day.replace(hour=0, minute=0, second=0, microsecond=0)
    to_time = day.replace(hour=23, minute=59, second=59, microsecond=999999)
    _df = get_sls_data_by_query(
        query=click_query,
        project="xianmu-front-end-log",
        logstore="xm-mall",
        from_time=from_time,
        to_time=to_time,
    )

    if not _df.empty:
        _df.drop(columns=["__source__", "__time__"], inplace=True)
        _df.to_sql(table_name, conn, if_exists='replace', index=False)
    conn.close()
    return _df

all_user_sku_click_df = pd.DataFrame()
start_date = datetime(2025, 1, 1)
end_date = datetime.now()
current_date = start_date
while current_date <= end_date:
    df = get_user_sku_click_of_date_from_sls(current_date,check_if_local_exist=False)
    all_user_sku_click_df = pd.concat([all_user_sku_click_df, df], ignore_index=True)
    current_date += timedelta(days=1)

all_user_sku_click_df.head(10)

即将获取数据: =====> 2025-01-01 00:00:00 2025-01-01 23:59:59.999999 xm-mall: 
type:cl and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  re
>=====数条数:1000
即将获取数据: =====> 2025-01-02 00:00:00 2025-01-02 23:59:59.999999 xm-mall: 
type:cl and pageName:/search/goods |
select   regexp_extract(sku_item, 'idx:(\d+)', 1) AS idx,
  re
>=====数条数:1000


,idx,name,sku,pid,pdid,ds,search_query,type,uid,page_name
0,null,null,null,null,null,20250101,安佳,cl,337640,/search/goods
1,null,加入购物车,16788463274,加购弹窗,1572,20250101,柠檬,cl,501851,/search/goods
2,null,null,null,null,null,20250101,青稞,cl,498537,/search/goods
3,null,null,null,null,null,20250101,青稞,cl,498537,/search/goods
4,null,立即购买,713137118123,加购弹窗,10279,20250101,青稞,cl,498537,/search/goods
5,1,佳农凤梨 毛重25-27斤/一级/7-8头,56065414871,goods,2879,20250101,凤梨,cl,453425,/search/goods
6,null,null,null,null,null,20250101,柠檬,cl,337723,/search/goods
7,0,蓝风车蓝米吉稀奶油 1L*12盒,null,goods,71,20250101,蓝风车蓝米吉稀奶油,cl,460667,/search/goods
8,34,安佳块状马苏里拉奶酪 10KG*2块,null,goods,156,20250101,安佳芝士条,cl,284610,/search/goods
9,1,小台农芒果冻肉 1KG*12包,780484374027,goods,3971,20250101,冷冻芒果,cl,311996,/search/goods
